# Init

In [1]:
from workspace import Workspace
from util import create_recipes

# workspace
workspace = Workspace(config_path=["config/base.j2", "config/layout.j2"])
core = workspace.components["core"]

❌ core connection failed @ 192.168.137.100
🔵 core simulation api enabled
✅ printer connected @ 192.168.137.102
❌ pipettor connection failed @ /dev/ttyUSB0
[Display] socket.io connected
[Display] sending initial snapshot (145 items)
[Display] Running at 60 fps


[gate] started, waiting for start...
,{"pose": [-0.04, 0.0, 0.00411, 0.0, 0.0, 0.0] ,"scale": [0.182, 0.118, 0.008] }
,{"pose": [-0.04, 0.0, 0.06767, 0.0, 0.0, 0.0] ,"scale": [0.182, 0.086, 0.118] }
,{"pose": [0.03480000085701463, -0.014395677835787405, 0.18781, -8.905192645881606, -44.823889094567726, -21.278072630486687] ,"scale": [0.216, 0.066, 0.092] }
,{"pose": [0.03810597726318987, -0.09464364055137173, 0.2214, -8.905192645881606, -44.823889094567726, -21.278072630486687] ,"scale": [0.06, 0.088, 0.06] }
,{"pose": [0.13177063658820778, 0.011558216096449778, 0.1536437127993202, 91.02449160436277, 34.45312162751387, -66.63087363802087] ,"scale": [0.328, 0.07, 0.052] }
,{"pose": [0.1023069422882787, 0.08170775937609231, 0.2641362731636175, 91.02449160436277, 34.45312162751387, -66.63087363802087] ,"scale": [0.058, 0.058, 0.058] }
,{"pose": [0.1513737921068753, -0.0847268958464061, 0.0413544260803164, -173.11032548424777, 34.39194653940296, 19.21273146488582] ,"scale": [0.052, 0.104, 

# workflow

In [2]:
def workflow_fn(*, workspace, core):
    """
    Pure workflow (NO wait_for_start, NO checkpoint).
    Runtime (rt.worker) handles waiting + pause/stop gating.
    """

    # ------------------------------------------------------------
    # Job config
    # ------------------------------------------------------------
    simulation = True
    speed_factor = 1

    tip_list = [f"{r}{c}" for r in "ABCDEFGH" for c in range(1, 13)]

    tube_list = [f"{r}{c}" for r in "AB" for c in range(1, 6)]
    num_processed_tube = 1
    falcon_rack_gravity_offset = 3

    cap_list = [f"{r}{c}" for r in "CD" for c in range(1, 6)]
    cap_offset = [0, 0, 111 - 2, 0, 0, 0]
    cap_gravity_offset = 1

    decapper_tool_tcp_z_offset = -1

    pipetting = True
    shake_travel = 7
    vol = 400  # ul
    immerse_depth = 20

    tool_rack_1_joint = [
        -34.628906, 46.625977, -78.486328, 1.010742,
        -57.854004, -32.717285, 263.0625, 0
    ]

    dry_run_count = 1
    printer_gravity_offset = 4

    inspection_frq = 4
    inspection_rot = 90

    tool_rack_0_joint = [
        -14.39209, 40.297852, -94.614258, -0.219727,
        -35.024414, -13.688965, 121.89375, 0
    ]

    # ------------------------------------------------------------
    # Build recipes (recipes MUST use workspace.rt.* for robot calls)
    # ------------------------------------------------------------
    rcp = create_recipes(workspace, core, speed_factor=speed_factor)

    # ------------------------------------------------------------
    # Simulation toggles
    # ------------------------------------------------------------
    if not simulation:
        core.simulation(False)
        workspace.components["pipettor"].simulation(False)
        workspace.components["printer"].simulation(False)

    # runtime handle (ok to use internally)
    rt = workspace.rt

    # ------------------------------------------------------------
    # Main loop
    # ------------------------------------------------------------
    for tip_index in tip_list:

        if pipetting:
            rcp["tool_rack_1"].pick()

            # gated by runtime (pause/stop safe)
            rt.jmove(
                joint=tool_rack_1_joint,
                vel=rcp["falcon_rack"].jmove_vaj[0] * rcp["falcon_rack"].speed_factor,
                accel=rcp["falcon_rack"].jmove_vaj[1] * rcp["falcon_rack"].speed_factor,
                jerk=rcp["falcon_rack"].jmove_vaj[2] * rcp["falcon_rack"].speed_factor,
            )

            rcp["tip_rack"].pick_tip(tip_index)

            for index in [num_processed_tube - 1, len(tube_list) - num_processed_tube]:
                aspirate_index = tube_list[index]
                dispense_index = tube_list[len(tube_list) - (index + 1)]

                rcp["falcon_pipepette"].immerse(anchor=aspirate_index, depth=immerse_depth)
                rcp["falcon_pipepette"].aspirate(vol=vol)
                rcp["falcon_pipepette"].retract(anchor=aspirate_index)

                rcp["falcon_pipepette"].immerse(anchor=dispense_index, depth=immerse_depth)
                rcp["falcon_pipepette"].dispense(vol=vol)
                rcp["falcon_pipepette"].retract(anchor=dispense_index)

            rcp["waste_bin"].eject_tip(shake_travel=shake_travel)
            rcp["tool_rack_1"].place()

        # pick gripper
        rcp["tool_rack_0"].pick()

        rt.jmove(
            joint=tool_rack_0_joint,
            vel=rcp["falcon_rack"].jmove_vaj[0] * rcp["falcon_rack"].speed_factor,
            accel=rcp["falcon_rack"].jmove_vaj[1] * rcp["falcon_rack"].speed_factor,
            jerk=rcp["falcon_rack"].jmove_vaj[2] * rcp["falcon_rack"].speed_factor,
        )

        # cap/print/inspect
        for index in range(num_processed_tube):
            tube_index = tube_list[index]
            cap_index = cap_list[index]

            rcp["falcon_rack"].pick_from(tube_index)
            rcp["decapper"].place()

            rcp["falcon_rack"].pick_from(cap_index)

            rcp["decapper"].cap(exit=False)
            rcp["decapper"].pick(approach=False, tool_tcp_z_offset=decapper_tool_tcp_z_offset)

            rcp["printer"].place(exit=False, gravity_offset=printer_gravity_offset)
            rcp["printer"].dry_run_spin(count=dry_run_count)
            rcp["printer"].pick(approach=False)

            rcp["inspector"].present(approach=False)
            for _ in range(inspection_frq):
                rcp["inspector"].rotate(rotation=inspection_rot)

            rcp["falcon_rack"].place_in(
                tube_index,
                gravity_offset=falcon_rack_gravity_offset,
                soft_approach=True,
            )

        # decap
        for index in range(num_processed_tube):
            tube_index = tube_list[index]
            cap_index = cap_list[index]

            rcp["falcon_rack"].pick_from(tube_index)

            rcp["decapper"].place(exit=False)
            rcp["decapper"].decap(approach=False)

            rcp["falcon_rack"].place_in(
                cap_index,
                offset=cap_offset,
                soft_approach=True,
                gravity_offset=cap_gravity_offset,
            )

            rcp["decapper"].pick(tool_tcp_z_offset=decapper_tool_tcp_z_offset)

            rcp["falcon_rack"].place_in(
                tube_index,
                gravity_offset=falcon_rack_gravity_offset,
                soft_approach=True,
            )

        # place gripper
        rt.jmove(
            joint=tool_rack_0_joint,
            vel=rcp["falcon_rack"].jmove_vaj[0] * rcp["falcon_rack"].speed_factor,
            accel=rcp["falcon_rack"].jmove_vaj[1] * rcp["falcon_rack"].speed_factor,
            jerk=rcp["falcon_rack"].jmove_vaj[2] * rcp["falcon_rack"].speed_factor,
        )

        rcp["tool_rack_0"].place()


# workflow thread

In [3]:
import threading
import traceback
from workspace.runtime import KillRequested, RTState


def start_job_thread(workflow_fn, *, workspace, core):
    """
    Gate-thread model:
      - waits on workspace.rt.wait_for_start()
      - each workspace.rt.start() runs workflow once
      - kill() exits the gate thread cleanly (no traceback)
    """

    rt = workspace.rt

    # ensure rt.robot_api points to the real robot API (SimulationAPI / Dorna), not Core
    if getattr(rt, "robot_api", None) is None or hasattr(getattr(rt, "robot_api", None), "robot_api"):
        rt.robot_api = core.robot_api

    def _gate_loop():
        print("[gate] started, waiting for start...", flush=True)

        while True:
            try:
                rt.wait_for_start()      # may raise KillRequested
                rt.mark_running()        # may raise KillRequested

                workflow_fn(workspace=workspace, core=core)

                rt.mark_idle()           # normal finish

            except KillRequested:
                print("[gate] killed -> exiting gate thread", flush=True)
                return

            except Exception as ex:
                rt.mark_error(ex)
                traceback.print_exc()
                # go idle after error unless killed
                if rt.state != RTState.KILLED:
                    rt.mark_idle()

    th = threading.Thread(target=_gate_loop, daemon=True)
    th.start()
    return th


# main loop

In [8]:
th = start_job_thread(workflow_fn, workspace=workspace, core=core)

In [ ]:
# when ready:
#workspace.rt.start()
#workspace.rt.pause()